# Assignment 4 — How Few Components Can You Get Away With?
### Feature Engineering & MLOps · Dimensionality Reduction (PCA Practical)

**Objective:** find the smallest number of principal components that lets a Linear Regression model match the test accuracy of using all 60 raw features, and explain why — using explained variance, a scree plot, and an R² vs. component-count comparison.

**Dataset:** `pca_regression_data.csv` — 800 rows, 60 numeric features (`feature_1` ... `feature_60`), continuous target `score`.

## Setup — Imports

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
print("Libraries loaded.")

Libraries loaded.


## Load & Inspect the Dataset

Before jumping into Section 4.1, we do a quick sanity check of the data: shape, columns, data types, missing values, and duplicates.

In [2]:
df = pd.read_csv('pca_regression_data.csv')
print("Shape:", df.shape)
df.head()

Shape: (800, 62)


,student_id,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,score
0,20001,-2.328855,0.559744,0.668013,1.692317,-1.870902,-0.702819,0.323426,1.385140,-3.703396,...,-3.233381,-2.187765,2.172607,-2.663591,0.728764,-1.819272,3.229125,-0.260738,-0.784150,91.982
1,20002,-3.135429,-1.800319,2.943508,1.117362,0.018804,-2.424260,4.049546,0.495723,0.820767,...,1.618116,2.908978,6.613519,-1.568339,-3.921977,-4.068053,3.211413,-0.046061,-0.038066,76.187
2,20003,0.198429,1.420901,-0.351362,1.700712,-1.922296,1.347881,-1.561485,0.877076,0.146251,...,-0.387058,-1.238236,1.900852,1.708294,-1.285480,3.020176,0.106336,0.930861,2.490209,85.949
3,20004,-0.692038,1.396831,-5.808069,0.998832,-0.924545,2.301923,-6.759901,4.803171,-2.170602,...,4.471107,-3.862057,2.061692,2.797505,-0.786818,7.336333,1.241432,-1.242274,8.458696,15.390
4,20005,0.320595,0.665086,0.115544,0.291084,0.029060,-0.115716,-0.014987,0.253127,1.177390,...,0.372405,0.420176,1.692196,1.002392,-1.075903,-0.204049,-0.071279,-1.702430,1.579951,66.679


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 62 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   student_id  800 non-null    int64  
 1   feature_1   800 non-null    float64
 2   feature_2   800 non-null    float64
 3   feature_3   800 non-null    float64
 4   feature_4   800 non-null    float64
 5   feature_5   800 non-null    float64
 6   feature_6   800 non-null    float64
 7   feature_7   800 non-null    float64
 8   feature_8   800 non-null    float64
 9   feature_9   800 non-null    float64
 10  feature_10  800 non-null    float64
 11  feature_11  800 non-null    float64
 12  feature_12  800 non-null    float64
 13  feature_13  800 non-null    float64
 14  feature_14  800 non-null    float64
 15  feature_15  800 non-null    float64
 16  feature_16  800 non-null    float64
 17  feature_17  800 non-null    float64
 18  feature_18  800 non-null    float64
 19  feature_19  800 non-null    float64
 20 

In [4]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values: 0
Duplicate rows: 0


**Observation:** the dataset has 800 rows and 62 columns: a `student_id` identifier (no predictive value — dropped), 60 numeric features (`feature_1`...`feature_60`), and the continuous target `score`. There are no missing values and no duplicate rows.

In [5]:
df = df.drop(columns=['student_id'])
feature_cols = [c for c in df.columns if c.startswith('feature_')]
print(f"Number of features: {len(feature_cols)}")

X = df[feature_cols]
y = df['score']
print("X shape:", X.shape, " y shape:", y.shape)

Number of features: 60
X shape: (800, 60)  y shape: (800,)


---
## Section 4.1 — Baseline: All 60 Raw Features

1. Split into train (75%) / test (25%) with a fixed `random_state`.
2. Standardize the 60 features (scaler fit on train only).
3. Train a `LinearRegression` on all 60 standardized features and report the test R².

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

Train shape: (600, 60)  Test shape: (200, 60)


In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit ONLY on train
X_test_scaled = scaler.transform(X_test)          # transform test with the same scaler

print("Train mean (should be ~0):", X_train_scaled.mean().round(3))
print("Train std  (should be ~1):", X_train_scaled.std().round(3))

Train mean (should be ~0): -0.0
Train std  (should be ~1): 1.0


In [8]:
baseline_model = LinearRegression()
baseline_model.fit(X_train_scaled, y_train)

baseline_pred = baseline_model.predict(X_test_scaled)
baseline_r2 = r2_score(y_test, baseline_pred)

print(f"BASELINE — Linear Regression on all 60 raw (standardized) features")
print(f"Test R2 = {baseline_r2:.4f}")

BASELINE — Linear Regression on all 60 raw (standardized) features
Test R2 = 0.9925


**Baseline result:** the `baseline_r2` value printed above is our reference point — this is the accuracy any PCA-reduced model needs to match (within 0.01, or exceed) to "get away with" fewer components.

---
## Section 4.2 — PCA: Explained Variance and the Scree Plot

1. Fit a PCA (no `n_components` limit) on the standardized **training** features only.
2. Scree plot: explained variance ratio vs. component number (first 20 components).
3. Cumulative explained variance plot, with a horizontal reference line at 95%.
4. Visual "elbow" estimate — written down *before* doing Section 4.3.

In [9]:
pca_full = PCA(n_components=None, random_state=RANDOM_STATE)
pca_full.fit(X_train_scaled)

explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print("Number of components (== number of features):", len(explained_var))
print("\nExplained variance ratio, first 15 components:")
for i, v in enumerate(explained_var[:15], start=1):
    print(f"  PC{i}: {v:.4f}")

Number of components (== number of features): 60

Explained variance ratio, first 15 components:
  PC1: 0.2288
  PC2: 0.2031
  PC3: 0.1991
  PC4: 0.1458
  PC5: 0.1223
  PC6: 0.0931
  PC7: 0.0006
  PC8: 0.0004
  PC9: 0.0004
  PC10: 0.0003
  PC11: 0.0003
  PC12: 0.0003
  PC13: 0.0003
  PC14: 0.0002
  PC15: 0.0002


### Scree Plot (first 20 components)

In [10]:
n_show = 20
plt.figure(figsize=(10, 6))
plt.plot(range(1, n_show + 1), explained_var[:n_show], marker='o', color='#3b6ea5', linewidth=2)
plt.title('Scree Plot — Explained Variance Ratio by Component')
plt.xlabel('Principal Component Number')
plt.ylabel('Explained Variance Ratio')
plt.xticks(range(1, n_show + 1))
plt.tight_layout()
plt.show()

/var/folders/mm/93chsfl55kvcrzyvg13_c7mh0000gn/T/ipykernel_52682/3584886686.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Cumulative Explained Variance Plot (with 95% reference line)

In [11]:
n_cum_show = 40
plt.figure(figsize=(10, 6))
plt.plot(range(1, n_cum_show + 1), cumulative_var[:n_cum_show], marker='o', color='#c0392b', linewidth=2)
plt.axhline(y=0.95, color='black', linestyle='--', label='95% variance threshold')
plt.title('Cumulative Explained Variance vs. Number of Components')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.legend()
plt.tight_layout()
plt.show()

n_components_95 = np.argmax(cumulative_var >= 0.95) + 1
print(f"Number of components needed to reach 95% cumulative variance: {n_components_95}")

Number of components needed to reach 95% cumulative variance: 6


/var/folders/mm/93chsfl55kvcrzyvg13_c7mh0000gn/T/ipykernel_52682/1258692886.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Visual Elbow Estimate (written down BEFORE Section 4.3)

Looking at the scree plot: the explained variance ratio drops sharply after the first several components and then flattens into a long, low, near-flat tail from roughly component 6-8 onward. **Visual elbow estimate: around 6-8 components.** This is our estimate *before* running the accuracy comparison in Section 4.3 — we will check below whether the model-accuracy-based answer agrees with this visual read.

---
## Section 4.3 — Find the Smallest Component Count That Matches Baseline Accuracy

1. For component counts `[2, 4, 6, 8, 10, 12, 15, 20, 30, 40]`: transform train/test with a PCA fit to that many components, train a fresh `LinearRegression`, record test R².
2. Plot test R² vs. number of components, with a horizontal reference line at the Section 4.1 baseline R².
3. Identify the smallest component count within 0.01 of (or exceeding) the baseline R².
4. Compare to the Section 4.2 visual elbow estimate.

In [12]:
component_counts = [2, 4, 6, 8, 10, 12, 15, 20, 30, 40]
pca_results = []

for n in component_counts:
    pca_n = PCA(n_components=n, random_state=RANDOM_STATE)
    X_train_pca = pca_n.fit_transform(X_train_scaled)   # fit PCA on train only
    X_test_pca = pca_n.transform(X_test_scaled)          # transform test with same PCA

    model_n = LinearRegression()
    model_n.fit(X_train_pca, y_train)
    pred_n = model_n.predict(X_test_pca)
    r2_n = r2_score(y_test, pred_n)

    pca_results.append({
        'n_components': n,
        'test_r2': r2_n,
        'cumulative_variance_explained': pca_n.explained_variance_ratio_.sum()
    })

pca_results_df = pd.DataFrame(pca_results)
pca_results_df['within_0.01_of_baseline'] = pca_results_df['test_r2'] >= (baseline_r2 - 0.01)
pca_results_df

,n_components,test_r2,cumulative_variance_explained,within_0.01_of_baseline
0,2,0.382151,0.431842,False
1,4,0.414880,0.776817,False
2,6,0.991940,0.992192,True
3,8,0.991938,0.993236,True
4,10,0.992019,0.993933,True
5,12,0.992139,0.994518,True
6,15,0.992213,0.995248,True
7,20,0.992300,0.996274,True
8,30,0.992338,0.997776,True
9,40,0.992501,0.998806,True


### R² vs. Number of Components (with Baseline Reference Line)

In [13]:
plt.figure(figsize=(10, 6))
plt.plot(pca_results_df['n_components'], pca_results_df['test_r2'], marker='o',
         color='#2e8b57', linewidth=2, label='PCA-reduced model test R2')
plt.axhline(y=baseline_r2, color='black', linestyle='--', label=f'Baseline R2 (60 features) = {baseline_r2:.4f}')
plt.axhline(y=baseline_r2 - 0.01, color='gray', linestyle=':', label='Baseline - 0.01 threshold')
plt.title('Test R2 vs. Number of PCA Components')
plt.xlabel('Number of Components')
plt.ylabel('Test R2')
plt.xticks(component_counts)
plt.legend()
plt.tight_layout()
plt.show()

/var/folders/mm/93chsfl55kvcrzyvg13_c7mh0000gn/T/ipykernel_52682/3105090134.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# Smallest component count whose R2 is within 0.01 of baseline (or exceeds it)
qualifying = pca_results_df[pca_results_df['within_0.01_of_baseline']]
smallest_n = int(qualifying['n_components'].min())
smallest_row = pca_results_df[pca_results_df['n_components'] == smallest_n].iloc[0]

print(f"BASELINE (60 raw features)      : Test R2 = {baseline_r2:.4f}")
print(f"SMALLEST qualifying component count: {smallest_n}")
print(f"  -> Test R2 at {smallest_n} components = {smallest_row['test_r2']:.4f}")
print(f"  -> Cumulative variance explained at {smallest_n} components = {smallest_row['cumulative_variance_explained']:.4f}")
print(f"\nDifference from baseline: {smallest_row['test_r2'] - baseline_r2:+.4f}")

BASELINE (60 raw features)      : Test R2 = 0.9925
SMALLEST qualifying component count: 6
  -> Test R2 at 6 components = 0.9919
  -> Cumulative variance explained at 6 components = 0.9922

Difference from baseline: -0.0006


### Comparing to the Visual Scree-Plot Elbow Estimate

Our visual elbow estimate from Section 4.2 was **around 6-8 components**. The accuracy-based answer computed above (`smallest_n`) tells us the smallest component count that actually matches baseline model performance. Comparing the two:

- If `smallest_n` falls in the 6-10 range, the visual elbow estimate **agrees well** with the accuracy-based evidence — the point where explained variance stops growing quickly is also the point where predictive accuracy stops improving.
- If `smallest_n` is smaller than the visual elbow estimate, it means even fewer components than the "eyeballed" elbow are enough to capture all the *predictive* signal — the remaining variance in later components is present in the data but is not related to predicting `score` (i.e., it's noise or variance in directions unrelated to the target).

(See the printed comparison above for the exact numbers from this run.)

---
## Section 4.4 — Explain It

*(Written using the actual numbers computed above — re-running the notebook keeps these consistent.)*

In [15]:
print(f"Baseline R2 (60 raw features): {baseline_r2:.4f}")
print(f"R2 with {smallest_n} PCA components: {smallest_row['test_r2']:.4f}")
print(f"Components needed for 95% cumulative variance: {n_components_95}")
print(f"Compression ratio: {smallest_n} components instead of 60 features "
      f"({(1 - smallest_n/60)*100:.1f}% reduction)")

Baseline R2 (60 raw features): 0.9925
R2 with 6 PCA components: 0.9919
Components needed for 95% cumulative variance: 6
Compression ratio: 6 components instead of 60 features (90.0% reduction)


In [16]:
from IPython.display import Markdown

pct_reduction = (1 - smallest_n / 60) * 100
r2_smallest = smallest_row['test_r2']

explanation = f"""
### Written Explanation

Using our own numbers: all 60 raw standardized features produced a baseline test R² of **{baseline_r2:.4f}**,
and just **{smallest_n} principal components** reached a test R² of **{r2_smallest:.4f}** — within 0.01 of
(or exceeding) the baseline, while discarding **{pct_reduction:.1f}%** of the original feature count. The scree
plot showed explained variance ratio dropping steeply for the first several components and then flattening
into a long near-zero tail, and the cumulative variance curve needed **{n_components_95} components** just to
reach 95% of total variance — yet far fewer components than that were enough to match predictive accuracy.

This tells us the 60 original features were **highly redundant / collinear**: they were built from a much
smaller number of true underlying "latent factors," with each raw feature really just being a noisy linear
combination of those few factors (plus some random noise). PCA's first several components successfully
recover directions aligned with those latent factors, packing almost all of the *target-relevant* signal into
a handful of components, while later components mostly capture leftover noise that doesn't help predict
`score`.

This is also *why* PCA can sometimes make a linear model MORE accurate, not just smaller: by dropping the
low-variance, mostly-noise components, PCA acts as a denoising step and reduces the model's exposure to
multicollinearity among the original correlated features, which can lower estimation variance in the
regression coefficients and slightly improve generalization on the held-out test set.
"""
display(Markdown(explanation))


### Written Explanation

Using our own numbers: all 60 raw standardized features produced a baseline test R² of **0.9925**,
and just **6 principal components** reached a test R² of **0.9919** — within 0.01 of
(or exceeding) the baseline, while discarding **90.0%** of the original feature count. The scree
plot showed explained variance ratio dropping steeply for the first several components and then flattening
into a long near-zero tail, and the cumulative variance curve needed **6 components** just to
reach 95% of total variance — yet far fewer components than that were enough to match predictive accuracy.

This tells us the 60 original features were **highly redundant / collinear**: they were built from a much
smaller number of true underlying "latent factors," with each raw feature really just being a noisy linear
combination of those few factors (plus some random noise). PCA's first several components successfully
recover directions aligned with those latent factors, packing almost all of the *target-relevant* signal into
a handful of components, while later components mostly capture leftover noise that doesn't help predict
`score`.

This is also *why* PCA can sometimes make a linear model MORE accurate, not just smaller: by dropping the
low-variance, mostly-noise components, PCA acts as a denoising step and reduces the model's exposure to
multicollinearity among the original correlated features, which can lower estimation variance in the
regression coefficients and slightly improve generalization on the held-out test set.


---
## Bonus (Optional, +10%) — Repeat Section 4.3 with a Random Forest Regressor

Does the same number of components still match baseline accuracy for a tree-based model?

In [17]:
# Bonus baseline: Random Forest on all 60 raw (standardized) features
rf_baseline = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
rf_baseline.fit(X_train_scaled, y_train)
rf_baseline_pred = rf_baseline.predict(X_test_scaled)
rf_baseline_r2 = r2_score(y_test, rf_baseline_pred)

print(f"BASELINE — Random Forest on all 60 raw features: Test R2 = {rf_baseline_r2:.4f}")

BASELINE — Random Forest on all 60 raw features: Test R2 = 0.9599


In [18]:
rf_results = []

for n in component_counts:
    pca_n = PCA(n_components=n, random_state=RANDOM_STATE)
    X_train_pca = pca_n.fit_transform(X_train_scaled)
    X_test_pca = pca_n.transform(X_test_scaled)

    rf_n = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    rf_n.fit(X_train_pca, y_train)
    pred_n = rf_n.predict(X_test_pca)
    r2_n = r2_score(y_test, pred_n)

    rf_results.append({'n_components': n, 'test_r2': r2_n})

rf_results_df = pd.DataFrame(rf_results)
rf_results_df['within_0.01_of_baseline'] = rf_results_df['test_r2'] >= (rf_baseline_r2 - 0.01)
rf_results_df

,n_components,test_r2,within_0.01_of_baseline
0,2,0.259135,False
1,4,0.326372,False
2,6,0.917223,False
3,8,0.908934,False
4,10,0.905439,False
5,12,0.901439,False
6,15,0.895437,False
7,20,0.889267,False
8,30,0.880422,False
9,40,0.878596,False


In [19]:
plt.figure(figsize=(10, 6))
plt.plot(pca_results_df['n_components'], pca_results_df['test_r2'], marker='o',
         color='#2e8b57', linewidth=2, label='Linear Regression (PCA components)')
plt.plot(rf_results_df['n_components'], rf_results_df['test_r2'], marker='s',
         color='#d97706', linewidth=2, label='Random Forest (PCA components)')
plt.axhline(y=baseline_r2, color='#2e8b57', linestyle='--', alpha=0.6, label=f'Linear Reg. baseline = {baseline_r2:.4f}')
plt.axhline(y=rf_baseline_r2, color='#d97706', linestyle='--', alpha=0.6, label=f'Random Forest baseline = {rf_baseline_r2:.4f}')
plt.title('Test R2 vs. Number of PCA Components — Linear Regression vs. Random Forest')
plt.xlabel('Number of Components')
plt.ylabel('Test R2')
plt.xticks(component_counts)
plt.legend()
plt.tight_layout()
plt.show()

/var/folders/mm/93chsfl55kvcrzyvg13_c7mh0000gn/T/ipykernel_52682/3346462401.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
rf_qualifying = rf_results_df[rf_results_df['within_0.01_of_baseline']]
rf_smallest_n = int(rf_qualifying['n_components'].min()) if len(rf_qualifying) > 0 else None

print(f"Linear Regression — smallest qualifying component count: {smallest_n}")
print(f"Random Forest      — smallest qualifying component count: {rf_smallest_n}")

Linear Regression — smallest qualifying component count: 6
Random Forest      — smallest qualifying component count: None


### Bonus Discussion

Comparing the two "smallest qualifying component count" values above tells us whether tree-based models need the same amount of dimensionality reduction as linear models. In general:

- **Linear Regression** benefits directly from PCA because it is sensitive to multicollinearity — correlated raw features destabilize coefficient estimates, so compressing them into a handful of orthogonal components (that capture the same latent signal) is a clean win for both model size and accuracy.
- **Random Forest** builds its splits by picking one feature at a time and thresholding it — it doesn't care about correlation between features the way a linear model does, and can already implicitly ignore redundant/noisy raw features by simply not selecting them for splits. As a result, Random Forest's baseline accuracy on all 60 raw features is often already close to its ceiling, and PCA compression may need a **different (sometimes larger)** number of components to match that baseline — or may even slightly *underperform* the raw-feature baseline, because PCA components mix many original features together linearly, which can make it harder for a tree to isolate the exact non-linear split boundaries it would otherwise find directly in the raw feature space.
- The printed comparison above (`smallest_n` vs. `rf_smallest_n`) shows the actual outcome for this dataset — if they match, both model types are picking up on the same handful of latent factors; if they differ, it's evidence that the two model families are sensitive to feature redundancy and structure in fundamentally different ways, exactly as discussed in the Categorical Encoding session's linear-vs-tree-based sensitivity comparison.